# Free Wan video generation for AutoUGC-TH (Kaggle / Colab GPU)

This notebook runs **ComfyUI + Wan** (an open-source Chinese image-to-video model) on a
**free GPU** and exposes it at a public URL. Point the AutoUGC-TH tool at that URL and your
product b-roll is generated for **$0**.

**Before running:**
1. Kaggle: create a Notebook, then in the right panel set **Accelerator = GPU (T4 x2 or P100)**
   and **Internet = On**. (Colab: Runtime -> Change runtime type -> GPU.)
2. Run the cells top to bottom.
3. The last cell prints a public URL like `https://something.trycloudflare.com`.
4. On your Mac, put it in `.env`:  `COMFYUI_URL=https://something.trycloudflare.com`
   and set `VIDEOGEN_PROVIDER=wan_comfyui` + `DRY_RUN=false`, then `make restart`.

> Model repos/filenames change over time. The variables at the top of the **Download models**
> cell are the one place to update if a download 404s — the rest is stable.

> Free GPUs are time-limited (Kaggle ~30h/week). Keep the notebook tab open while you generate;
> when it stops, just re-run these cells (you get a new URL — update `.env` and `make restart`).

## 1. Install ComfyUI + the nodes we need

In [ ]:
import os, subprocess, sys

ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
os.chdir(ROOT)

def sh(cmd):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=True)

if not os.path.isdir('ComfyUI'):
    sh('git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git')
os.chdir(f'{ROOT}/ComfyUI')
sh(f'{sys.executable} -m pip install -q -r requirements.txt')

# Custom nodes: GGUF loaders (quantized models fit a free GPU) + video output.
os.makedirs('custom_nodes', exist_ok=True)
nodes = {
    'ComfyUI-GGUF': 'https://github.com/city96/ComfyUI-GGUF.git',
    'ComfyUI-VideoHelperSuite': 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    'ComfyUI-WanVideoWrapper': 'https://github.com/kijai/ComfyUI-WanVideoWrapper.git',
}
for name, url in nodes.items():
    dst = f'custom_nodes/{name}'
    if not os.path.isdir(dst):
        sh(f'git clone --depth 1 {url} {dst}')
    req = f'{dst}/requirements.txt'
    if os.path.isfile(req):
        sh(f'{sys.executable} -m pip install -q -r {req}')
print('ComfyUI + nodes installed.')

## 2. Download the Wan model (GGUF quantized so it fits a free GPU)

Edit the four variables below if a download fails (find the current repo on Hugging Face by
searching e.g. `Wan2.2 I2V GGUF`). A ~Q4/Q5 GGUF is the sweet spot for a free T4/P100.

In [ ]:
from huggingface_hub import hf_hub_download
import shutil, os

# --- EDIT THESE if a file 404s (search Hugging Face for the current repo/filename) ---
UNET_REPO,  UNET_FILE  = 'QuantStack/Wan2.2-I2V-A14B-GGUF', 'wan2.2-i2v-a14b-Q4_K_M.gguf'
TEXT_REPO,  TEXT_FILE  = 'Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors'
VAE_REPO,   VAE_FILE   = 'Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/vae/wan_2.1_vae.safetensors'
# ------------------------------------------------------------------------------------

M = f'{ROOT}/ComfyUI/models'
for d in ('unet', 'clip', 'vae'):
    os.makedirs(f'{M}/{d}', exist_ok=True)

def fetch(repo, fname, dest_dir, dest_name):
    print(f'downloading {repo}/{fname} ...')
    p = hf_hub_download(repo_id=repo, filename=fname)
    dst = f'{dest_dir}/{dest_name}'
    if not os.path.exists(dst):
        shutil.copy(p, dst)
    print('  ->', dst)

fetch(UNET_REPO, UNET_FILE, f'{M}/unet', 'wan2.2_i2v.gguf')
fetch(TEXT_REPO, TEXT_FILE, f'{M}/clip', 'wan_text_encoder.safetensors')
fetch(VAE_REPO,  VAE_FILE,  f'{M}/vae',  'wan_vae.safetensors')
print('Models ready. (Filenames match the sample workflow in the tool.)')

## 3. Start ComfyUI + a public tunnel

This launches ComfyUI and a free Cloudflare tunnel. Watch the output for a
`https://<random>.trycloudflare.com` URL — that's your `COMFYUI_URL`. Leave this cell running.

In [ ]:
import subprocess, time, os, re, threading
os.chdir(f'{ROOT}/ComfyUI')

# Get cloudflared (no account needed for a quick tunnel).
if not os.path.isfile('cloudflared'):
    sh('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared')
    sh('chmod +x cloudflared')

# Launch ComfyUI on :8188.
comfy = subprocess.Popen([sys.executable, 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('ComfyUI starting on :8188 ...'); time.sleep(25)

# Open the tunnel and print the public URL.
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8188', '--no-autoupdate'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url = None
for line in tun.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break

print('\n' + '='*70)
print('  COMFYUI IS LIVE. Put this in your .env on the Mac:')
print(f'  COMFYUI_URL={url}')
print('  VIDEOGEN_PROVIDER=wan_comfyui   DRY_RUN=false   -> then: make restart')
print('='*70)
print('\nKeep this cell running while you generate videos.')

## Notes

- The tool's default workflow (`app/core/adapters/real/workflows/wan_i2v_template.json`) expects
  the model filenames this notebook writes (`wan2.2_i2v.gguf`, `wan_text_encoder.safetensors`,
  `wan_vae.safetensors`). If you customise it, keep them in sync — or export your own working
  ComfyUI graph as **Save (API Format)**, add the placeholder tokens
  (`__INPUT_IMAGE__ __PROMPT__ __NEGATIVE__ __FRAMES__ __WIDTH__ __HEIGHT__ __SEED__`), and set
  `COMFYUI_WORKFLOW_PATH` to it.
- **Node/class names** (`WanImageToVideo`, loaders) in the sample are illustrative. Open the
  ComfyUI UI at the tunnel URL, build a working Wan image-to-video graph once, and export it —
  that guarantees the node ids match your install.
- To go faster / skip free-tier limits later: run the same ComfyUI on a rented GPU
  (RunPod / Vast.ai, ~$0.20-0.50/hr) and use its URL as `COMFYUI_URL`.